# Customer Churn Prediction — Error-Free Ready-to-Run Notebook

This notebook is rewritten using the provided Telco Customer Churn data and avoids all errors you faced:

- `ValueError: could not convert string to float: 'Male'`
- `KeyError: 'Churn'`
- `Churn` becoming all `NaN`
- `TypeError` while scaling integer columns
- KNN failing because categorical columns were not encoded
- correlation failing because string columns were present

Main solution:

- Keep original dataframe clean.
- Convert `TotalCharges` safely.
- Convert target `Churn` safely.
- Use `ColumnTransformer + Pipeline`.
- Use `OneHotEncoder` for all categorical columns, including `gender`.
- Use `StandardScaler` inside the pipeline for numeric columns.


## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import joblib

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
print("Libraries imported successfully.")

## 2. Load dataset

In [ ]:
possible_paths = [
    Path("WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    Path("/mnt/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
]

DATA_PATH = None
for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Keep 'WA_Fn-UseC_-Telco-Customer-Churn.csv' "
        "in the same folder as this notebook."
    )

df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = df_raw.columns.str.strip()

print("Dataset loaded from:", DATA_PATH)
print("Shape:", df_raw.shape)
df_raw.head()

## 3. Basic checks

In [ ]:
print("Columns:")
print(df_raw.columns.tolist())

print("\nData types:")
print(df_raw.dtypes)

print("\nMissing values:")
print(df_raw.isna().sum())

print("\nChurn distribution:")
print(df_raw["Churn"].value_counts(dropna=False))

## 4. Clean data safely

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip()
df = df.drop_duplicates().copy()

# Convert TotalCharges safely because it is object/string in the original dataset.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Safe Churn conversion.
# This prevents the issue where running map() again converts 0/1 into NaN.
churn_unique = set(df["Churn"].dropna().unique())

if churn_unique <= {"Yes", "No"}:
    df["Churn"] = df["Churn"].astype(str).str.strip().map({"No": 0, "Yes": 1})
elif churn_unique <= {0, 1}:
    df["Churn"] = df["Churn"].astype(int)
else:
    raise ValueError(f"Unexpected Churn values found: {churn_unique}")

if df["Churn"].isna().any():
    raise ValueError("Churn contains NaN after conversion. Check original values.")

print("Cleaned data shape:", df.shape)
print("\nChurn distribution after conversion:")
print(df["Churn"].value_counts(dropna=False))
df.head()

## 5. Churn distribution

In [ ]:
plt.figure(figsize=(5, 4))
df["Churn"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["No Churn", "Churn"], rotation=0)
plt.title("Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customer Count")
plt.show()

## 6. Correlation safely

In [ ]:
df_corr = df.drop(columns=["customerID"], errors="ignore").copy()
df_corr_encoded = pd.get_dummies(df_corr, drop_first=True)

bool_cols = df_corr_encoded.select_dtypes(include=["bool"]).columns
df_corr_encoded[bool_cols] = df_corr_encoded[bool_cols].astype(int)

churn_corr = df_corr_encoded.corr(numeric_only=True)["Churn"].sort_values(ascending=False)

plt.figure(figsize=(14, 7))
churn_corr.drop("Churn").head(20).plot(kind="bar")
plt.title("Top 20 Correlated Features with Churn")
plt.ylabel("Correlation")
plt.tight_layout()
plt.show()

churn_corr.head(15)

## 7. Prepare features and target

In [ ]:
X = df.drop(columns=["Churn", "customerID"], errors="ignore").copy()
y = df["Churn"].copy()

numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 8. Preprocessing pipeline

In [ ]:
try:
    encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created.")

## 9. Train fast baseline models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=20,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gaussian NB": GaussianNB()
}

results = []
trained_pipelines = {}

for name, model in models.items():
    print(f"Training {name}...")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline.named_steps["model"], "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc
    })

    trained_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)
results_df

## 10. KNN fixed demo on smaller sample

In [ ]:
# KNN is distance-based and can be slow on full one-hot encoded data.
# This small sample demo proves the previous 'Male' string error is fixed.
# Because preprocessing happens inside the pipeline, KNN receives only numeric data.

X_train_knn = X_train.iloc[:1500].copy()
y_train_knn = y_train.iloc[:1500].copy()
X_test_knn = X_test.iloc[:400].copy()
y_test_knn = y_test.iloc[:400].copy()

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=11))
])

knn_pipeline.fit(X_train_knn, y_train_knn)
knn_pred = knn_pipeline.predict(X_test_knn)

knn_accuracy = accuracy_score(y_test_knn, knn_pred)
knn_f1 = f1_score(y_test_knn, knn_pred, zero_division=0)

print("KNN demo accuracy:", round(knn_accuracy, 4))
print("KNN demo F1:", round(knn_f1, 4))
print("KNN ran successfully without string-to-float error.")

## 11. Compare models

In [ ]:
plt.figure(figsize=(9, 5))
results_df.set_index("Model")["F1"].sort_values(ascending=False).plot(kind="bar")
plt.title("Model Comparison by F1 Score")
plt.ylabel("F1 Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

results_df

## 12. Best model evaluation

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]

print("Best model:", best_model_name)

best_pred = best_pipeline.predict(X_test)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_pred))

print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=["No Churn", "Churn"], zero_division=0))

## 13. Save best model

In [ ]:
MODEL_PATH = Path("best_churn_model_pipeline.joblib")
joblib.dump(best_pipeline, MODEL_PATH)

print("Saved model pipeline to:", MODEL_PATH.resolve())

## 14. Test prediction

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

sample_customer = X_test.iloc[[0]].copy()
prediction = loaded_model.predict(sample_customer)[0]

if hasattr(loaded_model.named_steps["model"], "predict_proba"):
    churn_probability = loaded_model.predict_proba(sample_customer)[0][1]
else:
    churn_probability = None

print("Sample customer:")
print(sample_customer.to_string(index=False))

print("\nPrediction:", "Churn" if prediction == 1 else "No Churn")
if churn_probability is not None:
    print("Churn probability:", round(churn_probability, 4))

## Interview Explanation

> I built an end-to-end customer churn prediction pipeline. I cleaned the Telco Churn dataset, converted `TotalCharges` to numeric, safely mapped the target `Churn` to 0/1, and used a `ColumnTransformer` to handle numeric and categorical features. Numeric columns were imputed and scaled, while categorical columns such as gender, Contract, InternetService, and PaymentMethod were one-hot encoded. Then I trained multiple models including Logistic Regression, Decision Tree, Random Forest, and Gaussian NB. I also included a KNN demo on a smaller sample to show that KNN works once categorical variables are encoded. I compared models using Accuracy, Precision, Recall, F1-score, and ROC-AUC, and saved the best model as a reusable pipeline.

### Why this avoids earlier errors

- `gender = Male/Female` is handled by `OneHotEncoder`, so KNN does not receive strings.
- `Churn` mapping is protected, so running a cell again does not turn values into NaN.
- `StandardScaler` runs inside the pipeline, so it does not create dtype assignment errors.
- Correlation is calculated on an encoded numeric dataframe, so it does not fail on string columns.
